In [0]:
df = spark.read.table("workspace.texi_data.taxi")

In [0]:
from pyspark.sql.functions import count
zones_per_borough = df.groupBy("Borough").agg(count("Zone").alias("total_zones"))
display(zones_per_borough)

In [0]:
from pyspark.sql.functions import count
location_count = df.groupBy("service_zone").agg(count("locationid").alias("loacation_count"))
display(location_count)

In [0]:
from pyspark.sql.functions import count
zones_per_borough = df.groupBy("Borough").agg(count("Zone").alias("total_zones"))\
    .orderBy("total_zones", ascending=False).limit(1)
display(zones_per_borough)

In [0]:
from pyspark.sql.functions import count
zones_per_borough = df.groupBy("Borough").agg(count("Zone").alias("total_zones"))\
    .where("total_zones > 10")
display(zones_per_borough)

In [0]:
from pyspark.sql.functions import count_distinct
zones_per_borough = df.groupBy("Borough").agg(count_distinct("service_zone").alias("total_zones"))
display(zones_per_borough)

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.window import Window
part = Window.partitionBy("Borough").orderBy("Zone")

df = df.withColumn("rn",f.row_number().over(part)).select("Borough","Zone","rn")
display(df)

In [0]:
df = spark.read.table("workspace.texi_data.taxi")
import pyspark.sql.functions as f
from pyspark.sql.window import Window
df=df.groupBy("Borough").agg(f.count("Zone").alias("total_zones"))
part = Window.orderBy(f.col('total_zones').desc())
df =df.withColumn("rank",f.rank().over(part)).select("Borough","total_zones","rank").orderBy("rank")
display(df)

In [0]:
df = spark.read.table("workspace.texi_data.taxi")
import pyspark.sql.functions as f
from pyspark.sql.window import Window
df=df.groupBy("Borough").agg(f.count("Zone").alias("total_zones"))
part = Window.orderBy(f.col('total_zones').desc())
df =df.withColumn("rank",f.dense_rank().over(part)).select("Borough","total_zones","rank").where(f.col("rank") == 2).orderBy("rank")
display(df)

In [0]:
df = spark.read.table("workspace.texi_data.taxi")
import pyspark.sql.functions as f
from pyspark.sql.window import Window
part = Window.orderBy(f.col('locationID').asc()).rowsBetween(Window.unboundedPreceding,Window.currentRow)
df =df.withColumn("count",f.count(f.col('zone')).over(part)).select("Borough","count")
display(df)

In [0]:
df = spark.read.table("workspace.texi_data.taxi")
import pyspark.sql.functions as f

df = df.groupBy("Borough").agg(f.min('zone').alias('First_Zone'),f.max('zone').alias('last_zone'))\
    .select('Borough','first_zone','last_zone').orderBy('Borough')
display(df)

In [0]:
df = spark.read.table("workspace.texi_data.taxi")
import pyspark.sql.functions as f

value = df.groupBy('Borough').agg(f.count('locationID').alias('c'))\
    .orderBy('c','Borough',ascending=False).limit(1)
display(value)
df = df.where(f.col('Borough') == value.collect()[0]['Borough'])\
    .select('zone','Borough')
display(df)

In [0]:
df = spark.read.table("workspace.texi_data.taxi")
import pyspark.sql.functions as f

value = df.groupBy('Borough').agg(f.count('zone').alias('zone_count')).where(f.col('zone_count') > 5)
display(value)

In [0]:
df = spark.read.table("workspace.texi_data.taxi")
import pyspark.sql.functions as f

value = df.where(f.col('Zone') == 'Newark Airport') \
          .select('service_zone') \
          .distinct()

service_value = value.collect()[0]['service_zone']

df2 = df.where(f.col('service_zone') == service_value)

display(df2)

In [0]:
df = spark.read.table("workspace.texi_data.taxi") 
import pyspark.sql.functions as f 

value = df.where(f.col('Zone')=='Boro Zone').select("Borough").distinct()
display(value)

In [0]:
df = spark.read.table("workspace.texi_data.taxi") 
import pyspark.sql.functions as f 
df = df.where(f.col("zone").like("%/%"))\
  .withColumn('zone', f.regexp_replace('Zone', '/', '-'))\
    .select('zone')
display(df)

In [0]:
df = spark.read.table("workspace.texi_data.taxi") 
import pyspark.sql.functions as f 
df = df.select(f.upper("borough"))
display(df)
               

In [0]:
df = spark.read.table("workspace.texi_data.taxi") 
import pyspark.sql.functions as f 
df = df.groupBy('zone').agg(f.count('zone').alias('zone_count'))\
    .where(f.col('zone_count')>1).select('Zone','zone_count')
display(df)

In [0]:
df = spark.read.table("workspace.texi_data.taxi")
from pyspark.sql.window import Window 
import pyspark.sql.functions as f 
df2= Window.orderBy(f.col('locationID').asc())
df= df.withColumn('next_ID', f.lead('locationID').over(df2))\
    .where(f.col('next_ID')- f.col('locationID')>1).select('locationID', 'next_ID')
display(df)

In [0]:
df = spark.read.table("workspace.texi_data.taxi")
import pyspark.sql.functions as f 
df = df.groupBy('Borough').agg(f.countDistinct('service_zone').alias('count'))\
    .where(f.col('count') == 1).select('Borough', 'count')
display(df)

In [0]:
import pyspark.sql.functions as f
df = spark.read.table("workspace.texi_data.taxi")
t = df.count()
df = df.groupBy("service_zone").count() \
           .withColumn("per", f.col("count") * 100.0 / t)
display(df)

In [0]:
import pyspark.sql.functions as f
df = spark.read.table("workspace.texi_data.taxi")
df2 = df.withColumn(
    "zone",
    f.when(f.col("service_zone") == "EWR", "Airport")
     .when(f.col("service_zone") == "Boro Zone", "City")
     .otherwise("Other")
)
display(df2)

In [0]:
import pyspark.sql.functions as f
df = spark.read.table("workspace.texi_data.taxi")
t = df.count()
df = (
    df.groupBy("Borough")
      .agg(f.count("Zone").alias("total_zone"))
      .withColumn("per_total_zone", f.round(f.col("total_zone") * 100 / t, 2))
)
display(df)

In [0]:
import pyspark.sql.functions as f
df = spark.read.table("workspace.texi_data.taxi")
t = df.count()
df =df.groupBy("Borough").agg(f.count("Zone").alias("total_zone"))
display(df)

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as f
df = spark.read.table("workspace.texi_data.taxi")
w = Window.orderBy(f.col("zone_count").desc())
df = (
    df.groupBy("Borough")
      .agg(f.count("zone").alias("zone_count"))
      .withColumn("rnk", f.rank().over(w))
      .filter(f.col("rnk") <= 3)
)
display(df)